Import and get lens

In [ ]:
#imports
import torch as th
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
from cnn_surgery.utils.load_dataset import load_dataset
dataset = 'mnist'
train, test, val = load_dataset(dataset, metrics_file='metrics_merged.csv', load_class_acc=True)

weights_train, outputs_train, configs_train = train
weights_test, outputs_test, configs_test = test
weights_val, outputs_val, configs_val = val

train_class_accuracies = outputs_train[:, -10:]
test_class_accuracies = outputs_test[:, -10:]
val_class_accuracies = outputs_val[:, -10:]

In [ ]:
from cnn_surgery.lenses.regressor_lens import get_regressor_lens, mse_mae
regressor_lens = get_regressor_lens(weights_train, train_class_accuracies, weights_val, val_class_accuracies, device="cpu")

In [ ]:
# test regressor on test set
loader = th.utils.data.DataLoader(
    th.utils.data.TensorDataset(th.tensor(weights_test, dtype=th.float32), th.tensor(test_class_accuracies, dtype=th.float32)),
    batch_size=1000, shuffle=False
)
mse, mae  = mse_mae(regressor_lens, loader, device="cpu")
print(f"Test set MSE: {mse:.4f}, MAE: {mae:.4f}")

LGTM

The probe learned well, but it did not learn exceptionally well. Perhaps for proper backprop we need lower MSE. Let's proceed anyway.

# Backprop wrt to input weight
Let's see if we can backpropagate wrt the input weights

In [ ]:
# get a model from test set
MODEL_IDX = 502
print(f"Model Index: {MODEL_IDX}")
model_weights = th.tensor(weights_test[MODEL_IDX], dtype=th.float32)
pred = regressor_lens(model_weights.unsqueeze(0)).squeeze(0) # type: ignore

plt.figure(figsize=(10, 6))
bar_width = 0.35
indices = np.arange(len(pred.detach().numpy()))

plt.bar(indices, pred.detach().numpy(), bar_width, label='Predicted', alpha=0.7)
plt.bar(indices + bar_width, test_class_accuracies[MODEL_IDX], bar_width, label='True', alpha=0.7)

plt.xlabel('Class Index')
plt.ylabel('Accuracy')
plt.title('Predicted vs True Class Accuracies')
plt.xticks(indices + bar_width / 2, indices)
plt.legend()
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

print("Training Params", configs_test.iloc[MODEL_IDX])
print("True Overal CNN Test Accuracy", outputs_test[MODEL_IDX][0])

In [ ]:
import plotly.graph_objects as go

fig = go.Figure()

fig.add_trace(go.Bar(
    x=indices,
    y=model_weights.detach().numpy(),
    name=f'Model Weights of {MODEL_IDX}',
    marker=dict(opacity=0.7)
))

fig.update_layout(
    title='Barplot of Model Weights',
    xaxis_title='Weight Index',
    yaxis_title='Weight Value',
    xaxis=dict(range=[4000, 4100]),
    legend=dict(title='Legend'),
    bargap=0.2,
    template='plotly_white'
)

fig.show()

In [ ]:
plt.figure(figsize=(10, 6))
plt.hist(model_weights.detach().numpy(), bins=50, alpha=0.7)
plt.xlabel('Weight Value')
plt.ylabel('Frequency')
plt.title('Histogram of Model Weights')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

In [ ]:
from cnn_surgery.utils.reconstruct_network import reconstruct_network
from cnn_surgery.utils.evaluate_per_class_accuracy import evaluate_classifier, load_testset_data

def test_network_accuracy(model_weights, activation):
    '''
    Returns: mean accuracy: float, per class accuracy: list[float]
    '''

    model = reconstruct_network(model_weights, activation)
    # we have to compile model before evaluating, func arguments are not used except metrics
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    x_test, y_test = load_testset_data(dataset)
    acc, class_acc = evaluate_classifier(model, x_test, y_test)

    return acc, class_acc


In [ ]:
simple_loss = lambda pred, true, target: pred[target]
# targeted_loss = lambda pred, true, target: pred[target] + ((true - pred) ** 2).mean()

def targeted_loss(pred, true, target):
    target_term = pred[target]

    # set target index to zero by multiplying with a mask
    mask = th.ones_like(pred, requires_grad=False)
    mask[target] = 0
    maintain_rest_term = (((true - pred) * mask) ** 2).mean()

    # print(f"Target Loss: {target_loss.item():.4f}, Maintain Rest Loss: {maintain_rest.item():.4f}")

    return target_term + maintain_rest_term

We are going to try to decrease Class 5 accuracy.

In [ ]:
from tqdm import tqdm
target_class = 5
STEPS = 100
MODEL_IDX = 502
step_size = 0.3

# convert model weights to tensor, require grad
doctored_weights = th.tensor(weights_test[MODEL_IDX], dtype=th.float32, requires_grad=True)

# initialize tensor to store cumulative gradients
cum_grads = th.zeros_like(doctored_weights)
preds_list = []
actual_accs_list = []

# freeze regressor lens weights
regressor_lens.eval()  # set regressor lens to evaluation mode
for param in regressor_lens.parameters():
    param.requires_grad = False
for i in tqdm(range(STEPS)):
    # forward pass
    # unsqueeze to make the forward pass work, squeeze to get rid of nonce batch dim
    actual_accs = test_network_accuracy(doctored_weights.detach().numpy(), configs_test.iloc[MODEL_IDX]['config.activation'])[1]
    pred = regressor_lens(doctored_weights.unsqueeze(0)).squeeze(0) # type: ignore
    preds_list.append(pred.detach().numpy())
    actual_accs_list.append(actual_accs)

    # I don't think this works? Because of course the true accuracy changes as we change the weights
    true = th.tensor(test_class_accuracies[MODEL_IDX], dtype=th.float32) 
    # compute loss for target class
    loss = targeted_loss(pred, true, target_class)  # minimize the accuracy for target class
    # backward pass
    loss.backward()
    # get gradients
    gradients = doctored_weights.grad
    # take small gradient step
    cum_grads += gradients.detach().numpy() * step_size
    doctored_weights.data -= step_size * gradients
    # zero gradients for next step
    doctored_weights.grad.zero_()

class_acc_before = test_class_accuracies[MODEL_IDX]
acc_after, class_acc_after = test_network_accuracy(doctored_weights.detach().numpy(), configs_test.iloc[MODEL_IDX]['config.activation'])

## Plot 1: Class accuracies before and after optimization, and new predicted
plt.figure(figsize=(10, 6))
indices = np.arange(len(class_acc_before))

bar_width = 0.25

plt.bar(indices, class_acc_before, bar_width, label='Before', alpha=0.7)
plt.bar(indices + bar_width, class_acc_after, bar_width, label='After', alpha=0.7)
plt.bar(indices + 2 * bar_width, pred.detach().numpy(), bar_width, label='New Predicted', alpha=0.7)

plt.xlabel('Class Index')
plt.ylabel('Accuracy')
plt.title(f'Class Accuracies Before, After Optimization, and New Predicted for Model {MODEL_IDX}\nattempting to unlearn Class {target_class}')
plt.xticks(indices + bar_width, indices)
plt.legend()
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

## Plot 2: Mean prediction error over optimization steps
diffs = [abs((np.array(preds_list[i]) - np.array(actual_accs_list[i])).mean()) for i in range(len(preds_list))]
plt.figure(figsize=(10, 6))
plt.plot(diffs)
plt.xlabel('Optimization Step')
plt.ylabel('Mean Prediction Error (Predicted - Actual)')
plt.title('Mean Prediction Error Over Optimization Steps')
plt.ylim(0, 0.1)
plt.show()


## Plot 3: Unlearning of 


## Plot 4: Cumulative gradients per weight index at the end
# # plot accumulated gradients
# plt.figure(figsize=(20, 6))
# indices = np.arange(len(cum_grads.detach().numpy()))
# plt.bar(indices, -cum_grads.detach().numpy(), alpha=0.7, color='orange', label='Cumulative Gradients')
# # original weights
# plt.bar(indices, weights_test[MODEL_IDX], alpha=0.7, color='blue', label='Original Weights')
# plt.xlabel('Weight Index')
# plt.ylabel('Cumulative Gradient Value')
# plt.title('Cumulative Gradients Over Optimization Steps')
# plt.xlim(-5,200)
# plt.legend()
# plt.show()

## Zeroing weights test

In [ ]:
## test gradient replacement/accumulation
target_class = 9

doctored_weights = th.tensor(weights_test[MODEL_IDX], dtype=th.float32, requires_grad=True)

regressor_lens.eval()
for param in regressor_lens.parameters():
    param.requires_grad = False

print('NOT ZEROING GRADS')
for i in range(10):
    # forward pass
    pred = regressor_lens(doctored_weights.unsqueeze(0)).squeeze(0) # type: ignore
    true = th.tensor(test_class_accuracies[MODEL_IDX], dtype=th.float32)
    loss = targeted_loss(pred, true, target_class)
    loss.backward()
    gradients = doctored_weights.grad
    print(f"Gradients mean: {gradients.detach().numpy().mean()}, std: {gradients.detach().numpy().std()}")

print('\nZEROING GRADS')
doctored_weights = th.tensor(weights_test[MODEL_IDX], dtype=th.float32, requires_grad=True)
for i in range(10):
    # forward pass
    pred = regressor_lens(doctored_weights.unsqueeze(0)).squeeze(0) # type: ignore
    true = th.tensor(test_class_accuracies[MODEL_IDX], dtype=th.float32)
    loss = targeted_loss(pred, true, target_class)
    loss.backward()
    gradients = doctored_weights.grad
    print(f"Gradients mean: {gradients.detach().numpy().mean()}, std: {gradients.detach().numpy().std()}")
    doctored_weights.grad.zero_()

## L2Reg stuff

In [ ]:
# give dataframe integer index
configs_test.reset_index(inplace=True)
configs_test

In [ ]:
# get all config.l2reg <= 1e-7
low_l2reg_index = configs_test[configs_test['config.l2reg'] <= 1e-7].index
# get only indices > val_size
low_l2reg_index = low_l2reg_index[low_l2reg_index > val_size]

In [ ]:
# test low l2reg models
low_l2reg_weights = th.tensor(test_weights[low_l2reg_index - val_size], dtype=th.float32)
low_l2reg_class_accuracies = th.tensor(test_class_accuracies[low_l2reg_index - val_size], dtype=th.float32)
low_l2reg_loader = th.utils.data.DataLoader(
    th.utils.data.TensorDataset(low_l2reg_weights, low_l2reg_class_accuracies),
    batch_size=1000, shuffle=False
)
mse, mae = mse_mae(regressor_lens, low_l2reg_loader)
print(f"Low L2Reg Test set MSE: {mse:.4f}, MAE: {mae:.4f}")

In [ ]:
# repeat for high l2reg models
high_l2reg_index = configs_test[configs_test['config.l2reg'] > 1e-3].index
# get only indices > val_size
high_l2reg_index = high_l2reg_index[high_l2reg_index > val_size]
# test high l2reg models
high_l2reg_weights = th.tensor(test_weights[high_l2reg_index - val_size], dtype=th.float32)
high_l2reg_class_accuracies = th.tensor(test_class_accuracies[high_l2reg_index - val_size], dtype=th.float32)
high_l2reg_loader = th.utils.data.DataLoader(
    th.utils.data.TensorDataset(high_l2reg_weights, high_l2reg_class_accuracies),
    batch_size=1000, shuffle=False
)
mse, mae = mse_mae(regressor_lens, high_l2reg_loader)
print(f"High L2Reg Test set MSE: {mse:.4f}, MAE: {mae:.4f}")

In [ ]:
len(high_l2reg_index)